# Ad Optimization Agent

**Completed submission**

This notebook builds a simulated advertising optimization agent. It uses campaign performance data, reasons about CPA compared with a target, selects one tool, executes the action, and records the result. It runs without an API key because the assignment uses simulated data.

**Agent loop:** Fetch data → Analyze → Decide action → Execute tool → Finish with a human-review log.


## 1. Imports and state definition

The `Campaign` class stores the state that moves through the workflow. `AdTools` represents actions that a real ad-platform API could perform.


In [ ]:
"""Working, no-API-key version of the Ad Optimization Agent workshop."""

from dataclasses import dataclass, asdict
from typing import Any


@dataclass
class Campaign:
    campaign_id: str
    budget: float
    spend: float
    impressions: int
    clicks: int
    conversions: int
    target_cpa: float
    ad_groups: dict[str, dict[str, Any]]

    @property
    def cpa(self) -> float:
        return self.spend / self.conversions if self.conversions else float("inf")

    @property
    def ctr(self) -> float:
        return self.clicks / self.impressions if self.impressions else 0.0


class AdTools:
    """Simulated tools that represent actions an ad platform could execute."""

    def __init__(self, campaign: Campaign):
        self.campaign = campaign
        self.history: list[str] = []

    def adjust_bid(self, ad_group_id: str, new_bid_amount: float) -> str:
        group = self.campaign.ad_groups[ad_group_id]
        group["Bid"] = round(new_bid_amount, 2)
        message = f"Bid for {ad_group_id} adjusted to ${new_bid_amount:.2f}."
        self.history.append(message)
        return message

    def pause_ad_group(self, ad_group_id: str) -> str:
        self.campaign.ad_groups[ad_group_id]["Status"] = "Paused"
        message = f"Ad group {ad_group_id} paused because CPA is above target."
        self.history.append(message)
        return message

    def request_new_creatives(self, campaign_id: str) -> str:
        message = f"New creative request submitted for {campaign_id}."
        self.history.append(message)
        return message


def fetch_campaign_data() -> Campaign:
    return Campaign(
        campaign_id="Campaign-2024-Q4", budget=1000, spend=850,
        impressions=50000, clicks=500, conversions=5, target_cpa=50,
        ad_groups={
            "AG-101": {"Status": "Active", "CPA": 25.0, "Bid": 2.50, "Conversions": 4},
            "AG-102": {"Status": "Active", "CPA": 450.0, "Bid": 3.00, "Conversions": 1},
        },
    )


def analyze_campaign(campaign: Campaign) -> dict[str, Any]:
    worst_id, worst = max(campaign.ad_groups.items(), key=lambda item: item[1]["CPA"])
    if worst["CPA"] >= campaign.target_cpa * 3 and worst["Conversions"] <= 1:
        recommendation = {"tool": "pause_ad_group", "args": {"ad_group_id": worst_id}}
        reason = f"{worst_id} has CPA ${worst['CPA']:.2f}, which is far above the ${campaign.target_cpa:.2f} target."
    elif campaign.cpa > campaign.target_cpa * 2:
        recommendation = {"tool": "request_new_creatives", "args": {"campaign_id": campaign.campaign_id}}
        reason = "Overall CPA is more than twice the target, so new creative testing is needed."
    else:
        recommendation = {"tool": "adjust_bid", "args": {"ad_group_id": worst_id, "new_bid_amount": max(0.50, worst["Bid"] * 0.90)}}
        reason = f"{worst_id} is the weakest active group, so its bid should be reduced cautiously."
    return {"recommendation": recommendation, "reason": reason}


def run_agent(max_steps: int = 3) -> dict[str, Any]:
    campaign = fetch_campaign_data()
    tools = AdTools(campaign)
    trace = [{"step": 1, "node": "fetch_data", "message": "Campaign metrics loaded."}]
    analysis = analyze_campaign(campaign)
    trace.append({"step": 2, "node": "agent_reasoning", "message": analysis["reason"], "recommendation": analysis["recommendation"]})
    tool_name = analysis["recommendation"]["tool"]
    result = getattr(tools, tool_name)(**analysis["recommendation"]["args"])
    trace.append({"step": 3, "node": "execute_tool", "message": result})
    trace.append({"step": 4, "node": "finish", "message": "Action completed and logged for human review."})
    return {"campaign": campaign, "trace": trace, "tool_history": tools.history, "analysis": analysis}


## 2. Build the LangGraph workflow

This is the same stateful workflow pattern as the starter code. The reasoning is deterministic so the notebook does not require a paid API key.


In [ ]:
!pip -q install langgraph


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    campaign: object
    analysis: dict
    trace: list
    final_message: str

def fetch_data_node(state):
    campaign = fetch_campaign_data()
    return {'campaign': campaign, 'trace': state.get('trace', []) + [{'node': 'fetch_data', 'message': 'Campaign metrics loaded.'}]}

def reasoning_node(state):
    analysis = analyze_campaign(state['campaign'])
    row = {'node': 'agent_reasoning', 'message': analysis['reason'], 'recommendation': analysis['recommendation']}
    return {'analysis': analysis, 'trace': state['trace'] + [row]}

def execute_tool_node(state):
    tools = AdTools(state['campaign'])
    recommendation = state['analysis']['recommendation']
    result = getattr(tools, recommendation['tool'])(**recommendation['args'])
    return {'trace': state['trace'] + [{'node': 'execute_tool', 'message': result}], 'final_message': result}

def finish_node(state):
    return {'trace': state['trace'] + [{'node': 'finish', 'message': 'Action completed and logged for human review.'}]}

workflow = StateGraph(AgentState)
workflow.add_node('fetch_data', fetch_data_node)
workflow.add_node('agent_reasoning', reasoning_node)
workflow.add_node('execute_tool', execute_tool_node)
workflow.add_node('finish', finish_node)
workflow.set_entry_point('fetch_data')
workflow.add_edge('fetch_data', 'agent_reasoning')
workflow.add_edge('agent_reasoning', 'execute_tool')
workflow.add_edge('execute_tool', 'finish')
workflow.add_edge('finish', END)
app = workflow.compile()
print('LangGraph workflow compiled successfully.')


## 3. Run the agent


In [ ]:
final_state = app.invoke({'campaign': None, 'analysis': {}, 'trace': [], 'final_message': ''})
for number, row in enumerate(final_state['trace'], start=1):
    print(f"{number}. {row['node']}: {row['message']}")
print('\nFinal recommendation:', final_state['final_message'])


## 2. Fetch and inspect campaign data


In [ ]:
campaign = fetch_campaign_data()
print('Campaign:', campaign.campaign_id)
print(f'Budget: ${campaign.budget:,.2f}')
print(f'Spend: ${campaign.spend:,.2f}')
print(f'Impressions: {campaign.impressions:,}')
print(f'Clicks: {campaign.clicks:,}')
print(f'Conversions: {campaign.conversions}')
print(f'Overall CPA: ${campaign.cpa:.2f}')
print(f'Target CPA: ${campaign.target_cpa:.2f}')
print(f'CTR: {campaign.ctr:.2%}')
for group_id, group in campaign.ad_groups.items():
    print(group_id, group)


## 3. Agent reasoning and tool execution

The agent compares each ad group with the target. Since AG-102 has a very high CPA and only one conversion, the agent chooses `pause_ad_group`.


In [ ]:
result = run_agent()
for row in result['trace']:
    print(f"{row['step']}. {row['node']}: {row['message']}")
print('\nFinal action:', result['tool_history'][-1])


## 4. Evaluation and conclusion

The campaign CPA is $170, compared with the $50 target. AG-102 has a $450 CPA and only one conversion, so pausing it is the safest simulated action. The action is logged for human review instead of silently changing a real advertising account.


In [ ]:
print('Tool history:')
for item in result['tool_history']:
    print('-', item)
assert result['tool_history'][-1].startswith('Ad group AG-102 paused')
print('\nSUCCESS: agent completed the fetch, reasoning, tool execution, and finish steps.')
